# Module 5.2: 参数高效微调 (PEFT)

## 1. 本章概览

### 📚 学习目标

1. **PEFT动机**：理解为什么需要参数高效微调
2. **LoRA**：掌握低秩适配的原理和实现
3. **Adapter**：理解瓶颈适配器的架构
4. **Prefix/Prompt Tuning**：掌握软提示方法
5. **方法比较**：了解各方法的优缺点和适用场景

### 🎯 核心问题

- 为什么不能总是全量微调？
- LoRA为什么有效？低秩假设的依据是什么？
- 不同PEFT方法的参数效率和性能如何权衡？
- 实际项目中如何选择PEFT方法？

### 🏢 业务场景

本章技术将应用于 `电商客服智能助理` 场景：
- 有限显存如何微调 7B 参数模型？→ LoRA/QLoRA 的参数高效策略
- 多个活动规则需要同时生效怎么切换？→ 适配器的即插即用架构

### 🗺️ 知识地图

```
全量微调的问题
    ↓
PEFT方法
├── LoRA (低秩适配)
├── Adapter (瓶颈适配器)
├── Prefix-Tuning (前缀调优)
├── Prompt-Tuning (提示调优)
└── 高级方法 (QLoRA, (IA)³, BitFit)
    ↓
方法比较与选择
    ↓
实际应用 (PEFT库, 多任务部署)
```

### ⏱️ 预计学习时间：3-4小时

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from collections import defaultdict
from typing import Dict, List, Tuple, Optional, Union
import time

np.random.seed(42)
torch.manual_seed(42)
print("✓ Libraries imported")
print(f"PyTorch version: {torch.__version__}")

## 2. 动机与背景

### 为什么需要PEFT？

**参数高效微调 (Parameter-Efficient Fine-Tuning, PEFT)** 是一类仅训练模型中少量参数（通常不到总参数的 10%）即可将预训练模型适配到下游任务的微调方法。在本章场景中，它是解决大模型全量微调在内存、存储和计算方面代价过高问题的核心方案。

**全量微调的问题**：

1. **内存开销巨大**
   - 模型参数 + 梯度 + 优化器状态 ≈ 4倍模型大小
   - LLaMA-7B全量微调需要 ~112GB GPU内存
   
2. **存储成本高**
   - 每个任务需要保存完整模型副本
   - 10个任务 = 10个完整模型

3. **训练时间长**
   - 更新所有参数需要更多计算
   - 大模型训练需要多GPU并行

4. **灾难性遗忘**（回顾 Module 5.1）
   - 全量微调可能破坏预训练知识
   - 在小数据集上容易过拟合

### PEFT的核心思想

**只训练少量参数，保持大部分预训练权重不变。**

```
全量微调：更新所有参数 (100%)
    ↓
PEFT：只更新少量参数 (0.1% - 10%)
    ↓
效果：接近全量微调的性能
```

### PEFT方法分类

```
PEFT Methods
├── 加法方法 (Additive)
│   ├── Adapter Layers    ← 插入小型模块
│   ├── Prefix-Tuning     ← 添加可训练前缀
│   └── Prompt-Tuning     ← 添加软提示
├── 选择性方法 (Selective)
│   └── BitFit            ← 只训练偏置
└── 重参数化方法 (Reparameterization)
    └── LoRA              ← 低秩分解
```

In [ ]:
# 🔬 Micro Practice: PEFT Efficiency Demonstration
# Goal: Compare full fine-tuning vs PEFT memory and parameters
# Expected outcome: Understand why PEFT is needed

class PretrainedModelSimulator:
    """
    Simulate a pre-trained model to demonstrate PEFT efficiency
    """
    
    @staticmethod
    def count_params(d_model: int, n_layers: int, n_heads: int) -> int:
        """Estimate parameter count for a Transformer model
        
        Args:
            d_model: Model dimension
            n_layers: Number of transformer layers
            n_heads: Number of attention heads
            
        Returns:
            Total parameter count
        """
        # Attention: Q, K, V, O projections
        attn_params = 4 * d_model * d_model * n_layers
        # FFN: 2 linear layers (d_model → 4*d_model → d_model)
        ffn_params = 2 * d_model * 4 * d_model * n_layers
        # Embeddings (assume vocab_size = 30000)
        embed_params = 30000 * d_model
        # LayerNorm
        ln_params = 4 * d_model * n_layers
        
        total = attn_params + ffn_params + embed_params + ln_params
        return total
    
    @staticmethod
    def estimate_memory(n_params: int, precision: str = 'fp32') -> float:
        """Estimate memory in GB
        
        Args:
            n_params: Number of parameters
            precision: Precision type ('fp32', 'fp16', 'int8', 'int4')
            
        Returns:
            Memory in GB
        """
        bytes_per_param = {'fp32': 4, 'fp16': 2, 'int8': 1, 'int4': 0.5}
        return n_params * bytes_per_param[precision] / (1024**3)

sim = PretrainedModelSimulator()

# Compare different model sizes
print("=== Pre-trained Model Sizes ===\n")
print(f"{'Model':<15} {'d_model':<10} {'Layers':<10} {'Params':<15} {'Memory (fp32)':<15}")
print("-" * 65)

models = [
    ('BERT-base', 768, 12, 8),
    ('BERT-large', 1024, 24, 16),
    ('GPT-2', 768, 12, 12),
    ('GPT-2 XL', 1600, 48, 25),
    ('LLaMA-7B', 4096, 32, 32),
    ('LLaMA-13B', 5120, 40, 40),
]

model_params = []
for name, d_model, n_layers, n_heads in models:
    params = sim.count_params(d_model, n_layers, n_heads)
    memory = sim.estimate_memory(params)
    model_params.append((name, params, memory))
    print(f"{name:<15} {d_model:<10} {n_layers:<10} {params/1e6:<15.1f}M {memory:<15.2f}GB")

print("\n" + "="*65)
print("\n=== Full Fine-tuning vs PEFT ===\n")

# For BERT-base
bert_params = sim.count_params(768, 12, 8)
bert_memory = sim.estimate_memory(bert_params)

# Full fine-tuning needs: model + gradients + optimizer states
full_ft_memory = bert_memory * 4  # model + grad + adam_m + adam_v

# LoRA (rank=8): only LoRA params need gradients/optimizer
lora_params = 12 * 2 * (768 * 8 + 8 * 768)  # 12 layers, Q and V
lora_memory = bert_memory + sim.estimate_memory(lora_params) * 3  # frozen model + lora grad + optimizer

print(f"BERT-base ({bert_params/1e6:.1f}M params):")
print(f"  Full fine-tuning:")
print(f"    Trainable params: {bert_params/1e6:.1f}M (100%)")
print(f"    Training memory: ~{full_ft_memory:.2f} GB")
print(f"  LoRA (rank=8):")
print(f"    Trainable params: {lora_params/1e6:.3f}M ({lora_params/bert_params*100:.2f}%)")
print(f"    Training memory: ~{lora_memory:.2f} GB")
print(f"    Memory savings: {(1-lora_memory/full_ft_memory)*100:.1f}%")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Model sizes
names = [m[0] for m in model_params]
params = [m[1]/1e6 for m in model_params]
memories = [m[2] for m in model_params]

axes[0].barh(names, params, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Parameters (Millions)', fontsize=12)
axes[0].set_title('Model Size Comparison', fontsize=14, weight='bold')
axes[0].set_xscale('log')
axes[0].grid(True, alpha=0.3, axis='x')

# Full FT vs PEFT memory
methods = ['Full FT\n(all params)', 'LoRA\n(rank=8)', 'LoRA\n(rank=4)', 'Prompt\n(length=20)']
lora4_params = 12 * 2 * (768 * 4 + 4 * 768)
prompt_params = 20 * 768
ft_memories = [
    full_ft_memory,
    bert_memory + sim.estimate_memory(lora_params) * 3,
    bert_memory + sim.estimate_memory(lora4_params) * 3,
    bert_memory + sim.estimate_memory(prompt_params) * 3,
]
colors = ['red', 'blue', 'dodgerblue', 'green']

bars = axes[1].bar(methods, ft_memories, color=colors, alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Training Memory (GB)', fontsize=12)
axes[1].set_title('Training Memory: Full FT vs PEFT', fontsize=14, weight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

for bar, mem in zip(bars, ft_memories):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'{mem:.2f}GB', ha='center', va='bottom', fontsize=9, weight='bold')

plt.tight_layout()
plt.show()

print("\n核心观察：")
print("- 全量微调需要存储模型、梯度和优化器状态（~4x模型大小）")
print("- PEFT只需要为少量参数存储梯度和优化器状态")
print("- 对于大模型（7B+），PEFT是唯一可行的微调方式")
print("\n✓ PEFT效率对比完成！")

## 3. LoRA (Low-Rank Adaptation)

**低秩适配 (Low-Rank Adaptation, LoRA)** 是一种基于重参数化的 PEFT 方法，通过将权重更新矩阵分解为两个低秩矩阵的乘积来大幅减少可训练参数量。在本章场景中，它是目前最流行且效果最好的 PEFT 方法之一，广泛应用于大语言模型的高效微调。

### 3.1 核心思想

**观察**：预训练模型的权重更新矩阵 $\Delta W$ 具有低秩特性。

**LoRA方法**：不直接更新权重 $W$，而是学习低秩分解：

$$W' = W + \Delta W = W + BA$$

其中：
- $W \in \mathbb{R}^{d \times k}$：原始权重（冻结）
- $B \in \mathbb{R}^{d \times r}$：上投影矩阵（可训练）
- $A \in \mathbb{R}^{r \times k}$：下投影矩阵（可训练）
- $r \ll \min(d, k)$：秩（通常 4-64）

**参数量对比**：
- 原始：$d \times k$ 个参数
- LoRA：$d \times r + r \times k = r(d + k)$ 个参数
- 当 $r = 4, d = k = 768$ 时：$768^2 = 589,824$ → $4 \times 768 \times 2 = 6,144$（减少 **99%**）

### 3.2 数学推导

In [ ]:
# 🔬 Micro Practice: Implement LoRA from Scratch
# Goal: Build LoRA layer from scratch
# Expected outcome: Understand low-rank adaptation mechanism

class LoRALayer(nn.Module):
    """
    LoRA (Low-Rank Adaptation) Layer
    
    Instead of updating W directly, we learn:
        W' = W + B @ A
    where B ∈ R^{d×r}, A ∈ R^{r×k}, and r << min(d, k)
    """
    
    def __init__(self, in_features: int, out_features: int, rank: int = 4, alpha: float = 1.0) -> None:
        """
        Args:
            in_features: Input dimension
            out_features: Output dimension
            rank: LoRA rank (low-rank dimension)
            alpha: Scaling factor
        """
        super(LoRALayer, self).__init__()
        
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        
        # Low-rank matrices
        # A: down-projection (in_features → rank)
        self.A = nn.Parameter(torch.randn(rank, in_features) * 0.01)
        
        # B: up-projection (rank → out_features)
        # Initialize B to zero so LoRA starts as identity
        self.B = nn.Parameter(torch.zeros(out_features, rank))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass: compute the LoRA delta
        
        Args:
            x: Input tensor (..., in_features)
        
        Returns:
            LoRA output (..., out_features)
        """
        # x @ A^T @ B^T * scaling
        return (x @ self.A.T @ self.B.T) * self.scaling

class LinearWithLoRA(nn.Module):
    """
    Linear layer with LoRA: W' = W + B @ A
    Original weight W is frozen, only A and B are trained
    """
    
    def __init__(self, linear_layer: nn.Linear, rank: int = 4, alpha: float = 1.0) -> None:
        """
        Args:
            linear_layer: Original nn.Linear layer (will be frozen)
            rank: LoRA rank
            alpha: Scaling factor
        """
        super(LinearWithLoRA, self).__init__()
        
        self.linear = linear_layer
        self.lora = LoRALayer(
            linear_layer.in_features,
            linear_layer.out_features,
            rank=rank,
            alpha=alpha
        )
        
        # Freeze original weights
        for param in self.linear.parameters():
            param.requires_grad = False
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """W'x = Wx + BAx"""
        return self.linear(x) + self.lora(x)
    
    def merge_weights(self) -> nn.Linear:
        """Merge LoRA weights into original for efficient inference"""
        with torch.no_grad():
            delta_W = (self.lora.B @ self.lora.A) * self.lora.scaling
            self.linear.weight.data += delta_W
        return self.linear

# Demo LoRA
print("=== LoRA Implementation ===\n")

# Create original linear layer
original = nn.Linear(128, 256, bias=False)
original_params = sum(p.numel() for p in original.parameters())

# Wrap with LoRA
lora_linear = LinearWithLoRA(original, rank=4, alpha=1.0)

# Count parameters
total_params = sum(p.numel() for p in lora_linear.parameters())
trainable_params = sum(p.numel() for p in lora_linear.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"Original layer: {128} → {256}")
print(f"LoRA rank: 4")
print(f"  Original parameters: {original_params:,} (frozen)")
print(f"  LoRA A (rank × in): {4 * 128} = {4 * 128}")
print(f"  LoRA B (out × rank): {256 * 4} = {256 * 4}")
print(f"  Total LoRA parameters: {trainable_params:,}")
print(f"  Parameter reduction: {(1 - trainable_params/original_params)*100:.1f}%")
print()

# Test forward pass
x = torch.randn(2, 10, 128)
out = lora_linear(x)
print(f"Input: {x.shape} → Output: {out.shape}")
print()

# Demonstrate weight merging
print("=== Weight Merging (for efficient inference) ===\n")

# Before merging: need both original + LoRA
print("Before merging:")
print(f"  Forward pass: W @ x + (B @ A) @ x * scaling")
print(f"  Two matrix multiplications needed")

# After merging: single matrix
merged = lora_linear.merge_weights()
print("\nAfter merging:")
print(f"  Forward pass: W' @ x  (where W' = W + B @ A * scaling)")
print(f"  Single matrix multiplication - same speed as original!")

# Verify outputs match
x_test = torch.randn(1, 5, 128)
out_merged = merged(x_test)
print(f"\n  Merged output shape: {out_merged.shape}")
print(f"  ✓ No additional inference cost after merging!")

print("\n✓ LoRA从零实现完成！")

In [ ]:
# 🔬 Micro Practice: LoRA Weight Decomposition Visualization
# Goal: Visualize how LoRA decomposes weight updates
# Expected outcome: Understand low-rank structure intuitively

print("=== LoRA Weight Decomposition Visualization ===\n")

# Create a simulated weight update matrix
d_model = 256
rank = 4

# Simulate a full-rank weight update (what full fine-tuning would learn)
np.random.seed(42)
full_update = np.random.randn(d_model, d_model) * 0.1

# LoRA decomposition: approximate with low-rank
A = np.random.randn(rank, d_model) * 0.1
B = np.random.randn(d_model, rank) * 0.1
lora_update = B @ A

# Compute approximation error
approx_error = np.linalg.norm(full_update - lora_update, 'fro') / np.linalg.norm(full_update, 'fro')

print(f"Matrix dimensions: {d_model} × {d_model}")
print(f"LoRA rank: {rank}")
print(f"Full update parameters: {d_model * d_model:,}")
print(f"LoRA parameters: {rank * d_model * 2:,}")
print(f"Parameter reduction: {(1 - rank * d_model * 2 / (d_model * d_model)) * 100:.1f}%")
print(f"Approximation error: {approx_error:.2%}\n")

# Visualize the decomposition
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Row 1: Full update vs LoRA approximation
im0 = axes[0, 0].imshow(full_update, cmap='RdBu', aspect='auto', vmin=-0.3, vmax=0.3)
axes[0, 0].set_title(f'Full Weight Update ΔW\n({d_model}×{d_model} = {d_model*d_model:,} params)', 
                     fontsize=11, weight='bold')
axes[0, 0].set_xlabel('Input Dimension')
axes[0, 0].set_ylabel('Output Dimension')
plt.colorbar(im0, ax=axes[0, 0])

im1 = axes[0, 1].imshow(lora_update, cmap='RdBu', aspect='auto', vmin=-0.3, vmax=0.3)
axes[0, 1].set_title(f'LoRA Approximation B@A\n({rank * d_model * 2:,} params)', 
                     fontsize=11, weight='bold')
axes[0, 1].set_xlabel('Input Dimension')
axes[0, 1].set_ylabel('Output Dimension')
plt.colorbar(im1, ax=axes[0, 1])

error_map = np.abs(full_update - lora_update)
im2 = axes[0, 2].imshow(error_map, cmap='Reds', aspect='auto')
axes[0, 2].set_title(f'Approximation Error\n(Relative: {approx_error:.1%})', 
                     fontsize=11, weight='bold')
axes[0, 2].set_xlabel('Input Dimension')
axes[0, 2].set_ylabel('Output Dimension')
plt.colorbar(im2, ax=axes[0, 2])

# Row 2: LoRA decomposition components
im3 = axes[1, 0].imshow(B, cmap='RdBu', aspect='auto', vmin=-0.3, vmax=0.3)
axes[1, 0].set_title(f'LoRA Matrix B\n({d_model}×{rank})', fontsize=11, weight='bold')
axes[1, 0].set_xlabel(f'Rank Dimension ({rank})')
axes[1, 0].set_ylabel(f'Output Dimension ({d_model})')
plt.colorbar(im3, ax=axes[1, 0])

im4 = axes[1, 1].imshow(A, cmap='RdBu', aspect='auto', vmin=-0.3, vmax=0.3)
axes[1, 1].set_title(f'LoRA Matrix A\n({rank}×{d_model})', fontsize=11, weight='bold')
axes[1, 1].set_xlabel(f'Input Dimension ({d_model})')
axes[1, 1].set_ylabel(f'Rank Dimension ({rank})')
plt.colorbar(im4, ax=axes[1, 1])

# Singular value comparison
U_full, s_full, Vt_full = np.linalg.svd(full_update, full_matrices=False)
U_lora, s_lora, Vt_lora = np.linalg.svd(lora_update, full_matrices=False)

axes[1, 2].plot(s_full[:50], 'o-', label='Full Update', linewidth=2, markersize=4)
axes[1, 2].plot(s_lora[:50], 's-', label='LoRA Approx', linewidth=2, markersize=4)
axes[1, 2].axvline(x=rank, color='red', linestyle='--', linewidth=2, label=f'LoRA Rank={rank}')
axes[1, 2].set_xlabel('Singular Value Index', fontsize=10)
axes[1, 2].set_ylabel('Singular Value Magnitude', fontsize=10)
axes[1, 2].set_title('Singular Value Spectrum', fontsize=11, weight='bold')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)
axes[1, 2].set_yscale('log')

plt.tight_layout()
plt.show()

print("核心观察：")
print("- LoRA通过低秩分解B@A近似完整的权重更新ΔW")
print("- 只需存储两个小矩阵B和A，而不是完整的ΔW")
print("- 低秩假设：权重更新的主要信息集中在少数几个主成分上")
print("- 奇异值图显示：前几个奇异值包含了大部分信息")
print("\n✓ LoRA权重分解可视化完成！")

In [ ]:
# 🔬 Micro Practice: Apply LoRA to Transformer Attention
# Goal: See LoRA in a realistic setting
# Expected outcome: Understand practical LoRA usage

class MultiHeadAttentionWithLoRA(nn.Module):
    """
    Multi-Head Attention with LoRA applied to Q and V projections
    """
    
    def __init__(self, d_model: int, num_heads: int, lora_rank: int = 4, lora_alpha: float = 1.0) -> None:
        super(MultiHeadAttentionWithLoRA, self).__init__()
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        
        # Original projections (frozen)
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        
        # LoRA for Q projection
        self.lora_q = LoRALayer(d_model, d_model, rank=lora_rank, alpha=lora_alpha)
        
        # LoRA for V projection
        self.lora_v = LoRALayer(d_model, d_model, rank=lora_rank, alpha=lora_alpha)
        
        # Freeze original weights
        for param in [self.W_q, self.W_k, self.W_v, self.W_o]:
            for p in param.parameters():
                p.requires_grad = False
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, _ = x.shape
        
        # Q with LoRA
        Q = self.W_q(x) + self.lora_q(x)
        
        # K without LoRA (frozen)
        K = self.W_k(x)
        
        # V with LoRA
        V = self.W_v(x) + self.lora_v(x)
        
        # Reshape for multi-head
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn_weights = torch.softmax(scores, dim=-1)
        attn_output = torch.matmul(attn_weights, V)
        
        # Reshape back
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        
        return self.W_o(attn_output)

# Demo LoRA on attention
print("=== LoRA Applied to Multi-Head Attention ===\n")

d_model = 256
num_heads = 4
lora_rank = 4

attn_lora = MultiHeadAttentionWithLoRA(d_model, num_heads, lora_rank=lora_rank)

# Count parameters
total_params = sum(p.numel() for p in attn_lora.parameters())
trainable_params = sum(p.numel() for p in attn_lora.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"Model dimensions: d_model={d_model}, heads={num_heads}")
print(f"LoRA rank: {lora_rank}")
print()
print(f"Parameter breakdown:")
print(f"  Original attention: {frozen_params:,} (frozen)")
print(f"  LoRA Q: {sum(p.numel() for p in attn_lora.lora_q.parameters()):,}")
print(f"  LoRA V: {sum(p.numel() for p in attn_lora.lora_v.parameters()):,}")
print(f"  Total trainable: {trainable_params:,}")
print(f"  Trainable ratio: {trainable_params/total_params*100:.2f}%")
print()

# Test forward pass
x = torch.randn(2, 10, d_model)
out = attn_lora(x)
print(f"Input: {x.shape} → Output: {out.shape}")

# Compare different ranks
print("\n=== LoRA Rank vs Parameters ===\n")
print(f"{'Rank':<10} {'LoRA Params':<15} {'Total Params':<15} {'Ratio':<10}")
print("-" * 50)

for rank in [1, 2, 4, 8, 16, 32, 64]:
    lora_params = 2 * (d_model * rank + rank * d_model)  # Q and V
    total = 4 * d_model * d_model  # Original attention
    ratio = lora_params / total * 100
    print(f"{rank:<10} {lora_params:<15,} {total:<15,} {ratio:<10.2f}%")

# Visualize rank vs parameters
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ranks = [1, 2, 4, 8, 16, 32, 64]
lora_params_list = [2 * (d_model * r + r * d_model) for r in ranks]
ratios = [p / (4 * d_model * d_model) * 100 for p in lora_params_list]

axes[0].plot(ranks, lora_params_list, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('LoRA Rank', fontsize=12)
axes[0].set_ylabel('Trainable Parameters', fontsize=12)
axes[0].set_title('LoRA Rank vs Parameters', fontsize=14, weight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(ranks, ratios, 'ro-', linewidth=2, markersize=8)
axes[1].set_xlabel('LoRA Rank', fontsize=12)
axes[1].set_ylabel('Parameter Ratio (%)', fontsize=12)
axes[1].set_title('LoRA Rank vs Parameter Ratio', fontsize=14, weight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n观察：")
print("- rank=4时只需0.78%的参数")
print("- rank=8是常用默认值，平衡效率和性能")
print("- 通常只对Q和V应用LoRA（K的影响较小）")
print("\n✓ LoRA应用于Transformer完成！")

In [ ]:
# 🔬 Micro Practice: Actual Performance Benchmarking
# Goal: Measure real training time and memory usage
# Expected outcome: Understand practical efficiency gains

class PerformanceBenchmark:
    """Benchmark training performance of different PEFT methods"""
    
    @staticmethod
    def create_simple_model(input_dim: int, hidden_dim: int, output_dim: int) -> nn.Module:
        """Create a simple model for benchmarking"""
        return nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    @staticmethod
    def benchmark_training_step(model: nn.Module, optimizer: torch.optim.Optimizer, 
                               data: torch.Tensor, target: torch.Tensor, 
                               num_steps: int = 100) -> Tuple[float, int]:
        """Benchmark training time and memory
        
        Args:
            model: Model to train
            optimizer: Optimizer
            data: Input data
            target: Target labels
            num_steps: Number of training steps
            
        Returns:
            Tuple of (time_per_step_ms, trainable_params)
        """
        model.train()
        criterion = nn.CrossEntropyLoss()
        
        # Warmup
        for _ in range(5):
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
        
        # Actual benchmark
        start_time = time.time()
        for _ in range(num_steps):
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
        
        elapsed = time.time() - start_time
        time_per_step = (elapsed / num_steps) * 1000  # Convert to ms
        
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        
        return time_per_step, trainable_params

print("=== Actual Performance Benchmarking ===\n")
print("Comparing training speed of Full Fine-tuning vs PEFT methods\n")

# Setup
input_dim, hidden_dim, output_dim = 128, 512, 10
batch_size = 32
num_steps = 100

# Create dummy data
data = torch.randn(batch_size, input_dim)
target = torch.randint(0, output_dim, (batch_size,))

results = {}

# 1. Full Fine-tuning
print("1. Benchmarking Full Fine-tuning...")
model_full = PerformanceBenchmark.create_simple_model(input_dim, hidden_dim, output_dim)
optimizer_full = torch.optim.AdamW(model_full.parameters(), lr=1e-3)
time_full, params_full = PerformanceBenchmark.benchmark_training_step(
    model_full, optimizer_full, data, target, num_steps
)
results['Full FT'] = {'time': time_full, 'params': params_full}
print(f"   Time per step: {time_full:.2f}ms, Trainable params: {params_full:,}\n")

# 2. LoRA (simulated by training subset of parameters)
print("2. Benchmarking LoRA (rank=8)...")
model_lora = PerformanceBenchmark.create_simple_model(input_dim, hidden_dim, output_dim)
# Freeze most parameters, only train a small subset (simulating LoRA)
for name, param in model_lora.named_parameters():
    param.requires_grad = False
# Add small trainable adapters (simulating LoRA matrices)
lora_params = []
for i in range(3):  # 3 layers
    lora_A = nn.Parameter(torch.randn(8, hidden_dim) * 0.01)
    lora_B = nn.Parameter(torch.zeros(hidden_dim, 8))
    lora_params.extend([lora_A, lora_B])
    setattr(model_lora, f'lora_A_{i}', lora_A)
    setattr(model_lora, f'lora_B_{i}', lora_B)

optimizer_lora = torch.optim.AdamW(lora_params, lr=1e-3)
time_lora, params_lora = PerformanceBenchmark.benchmark_training_step(
    model_lora, optimizer_lora, data, target, num_steps
)
results['LoRA (r=8)'] = {'time': time_lora, 'params': params_lora}
print(f"   Time per step: {time_lora:.2f}ms, Trainable params: {params_lora:,}\n")

# 3. Adapter (simulated)
print("3. Benchmarking Adapter (bottleneck=32)...")
model_adapter = PerformanceBenchmark.create_simple_model(input_dim, hidden_dim, output_dim)
for param in model_adapter.parameters():
    param.requires_grad = False
# Add adapter parameters
adapter_params = []
for i in range(2):  # 2 adapters
    down = nn.Parameter(torch.randn(32, hidden_dim) * 0.01)
    up = nn.Parameter(torch.randn(hidden_dim, 32) * 0.01)
    adapter_params.extend([down, up])
    setattr(model_adapter, f'adapter_down_{i}', down)
    setattr(model_adapter, f'adapter_up_{i}', up)

optimizer_adapter = torch.optim.AdamW(adapter_params, lr=1e-3)
time_adapter, params_adapter = PerformanceBenchmark.benchmark_training_step(
    model_adapter, optimizer_adapter, data, target, num_steps
)
results['Adapter (b=32)'] = {'time': time_adapter, 'params': params_adapter}
print(f"   Time per step: {time_adapter:.2f}ms, Trainable params: {params_adapter:,}\n")

# Summary
print("="*70)
print("\n=== Performance Summary ===\n")
print(f"{'Method':<20} {'Time/Step (ms)':<18} {'Trainable Params':<18} {'Speedup':<12}")
print("-"*70)

for method, data in results.items():
    speedup = time_full / data['time']
    print(f"{method:<20} {data['time']:<18.2f} {data['params']:<18,} {speedup:<12.2f}x")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

methods = list(results.keys())
times = [results[m]['time'] for m in methods]
params = [results[m]['params'] for m in methods]
speedups = [time_full / results[m]['time'] for m in methods]
colors = ['red', 'blue', 'green']

# Training time
bars1 = axes[0].bar(methods, times, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Time per Step (ms)', fontsize=11)
axes[0].set_title('Training Speed Comparison', fontsize=12, weight='bold')
axes[0].grid(True, alpha=0.3, axis='y')
for bar, t in zip(bars1, times):
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height,
                f'{t:.1f}ms', ha='center', va='bottom', fontsize=9, weight='bold')

# Trainable parameters
bars2 = axes[1].bar(methods, params, color=colors, alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Trainable Parameters', fontsize=11)
axes[1].set_title('Parameter Efficiency', fontsize=12, weight='bold')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3, axis='y')
for bar, p in zip(bars2, params):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'{p:,}', ha='center', va='bottom', fontsize=8, weight='bold')

# Speedup
bars3 = axes[2].bar(methods, speedups, color=colors, alpha=0.7, edgecolor='black')
axes[2].set_ylabel('Speedup Factor', fontsize=11)
axes[2].set_title('Training Speedup vs Full FT', fontsize=12, weight='bold')
axes[2].axhline(y=1.0, color='red', linestyle='--', linewidth=2, label='Baseline (Full FT)')
axes[2].grid(True, alpha=0.3, axis='y')
axes[2].legend()
for bar, s in zip(bars3, speedups):
    height = bar.get_height()
    axes[2].text(bar.get_x() + bar.get_width()/2., height,
                f'{s:.2f}x', ha='center', va='bottom', fontsize=9, weight='bold')

plt.tight_layout()
plt.show()

print("\n核心观察：")
print("- PEFT方法训练速度更快（更少参数需要更新）")
print("- LoRA和Adapter的训练时间相近，都比全量微调快")
print("- 参数量减少90%+，但训练速度提升相对温和（因为前向传播仍需计算）")
print("- 实际应用中，内存节省是PEFT的最大优势")
print("\n✓ 实际性能对比完成！")

## 4. Adapter Layers

**适配器 (Adapter)** 是一种基于加法的 PEFT 方法，通过在 Transformer 层之间插入小型瓶颈网络模块，仅训练这些新增模块的参数来实现任务适配。在本章场景中，它提供了一种模块化的微调方案，不同任务可以共享同一预训练模型、各自维护独立的 Adapter 模块。

### 4.1 Adapter架构

**核心思想**：在Transformer层之间插入小型瓶颈模块，只训练这些模块。

**Adapter结构**：

$$\text{Adapter}(x) = x + W_{\text{up}} \cdot \text{ReLU}(W_{\text{down}} \cdot \text{LN}(x))$$

其中：
- $W_{\text{down}} \in \mathbb{R}^{d \times r}$：下投影（压缩）
- $W_{\text{up}} \in \mathbb{R}^{r \times d}$：上投影（恢复）
- $r \ll d$：瓶颈维度
- 残差连接确保初始时不影响原始模型

**插入位置**：

```
Input → Self-Attention → [Adapter] → FFN → [Adapter] → Output
```

**参数量**：每个Adapter约 $2 \times d \times r$ 个参数

**优势**：
- 模块化设计，易于添加和移除
- 不同任务使用不同Adapter
- 原始模型完全不变

In [ ]:
# 🔬 Micro Practice: Implement Adapter Layers
# Goal: Build Adapter module from scratch
# Expected outcome: Understand bottleneck adapter architecture

class AdapterLayer(nn.Module):
    """
    Adapter Layer: Bottleneck architecture with residual connection
    
    Architecture:
        x → LayerNorm → Down-project → ReLU → Up-project → + x (residual)
    """
    
    def __init__(self, hidden_dim, bottleneck_dim):
        """
        Args:
            hidden_dim: Original hidden dimension
            bottleneck_dim: Bottleneck dimension (much smaller)
        """
        super(AdapterLayer, self).__init__()
        
        self.hidden_dim = hidden_dim
        self.bottleneck_dim = bottleneck_dim
        
        # Down-projection: hidden_dim → bottleneck_dim
        self.down_proj = nn.Linear(hidden_dim, bottleneck_dim)
        
        # Non-linearity
        self.activation = nn.ReLU()
        
        # Up-projection: bottleneck_dim → hidden_dim
        self.up_proj = nn.Linear(bottleneck_dim, hidden_dim)
        
        # Layer normalization
        self.layer_norm = nn.LayerNorm(hidden_dim)
        
        # Initialize up_proj to near-zero for stable training
        nn.init.zeros_(self.up_proj.weight)
        nn.init.zeros_(self.up_proj.bias)
    
    def forward(self, x):
        """
        Forward pass with residual connection
        
        Args:
            x: Input tensor (batch_size, seq_len, hidden_dim)
        
        Returns:
            Output tensor (same shape as input)
        """
        residual = x
        
        # Adapter path
        x = self.layer_norm(x)
        x = self.down_proj(x)      # Compress
        x = self.activation(x)      # Non-linearity
        x = self.up_proj(x)         # Expand back
        
        # Residual connection
        return residual + x

class TransformerWithAdapters(nn.Module):
    """
    Simplified Transformer block with Adapter layers inserted
    """
    
    def __init__(self, hidden_dim, num_heads, bottleneck_dim=32):
        super(TransformerWithAdapters, self).__init__()
        
        self.hidden_dim = hidden_dim
        
        # Original transformer components (frozen)
        self.self_attention = nn.MultiheadAttention(hidden_dim, num_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.ReLU(),
            nn.Linear(hidden_dim * 4, hidden_dim)
        )
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        
        # Adapter layers (trainable)
        self.adapter_after_attn = AdapterLayer(hidden_dim, bottleneck_dim)
        self.adapter_after_ffn = AdapterLayer(hidden_dim, bottleneck_dim)
    
    def forward(self, x):
        # Self-attention + adapter
        attn_out, _ = self.self_attention(x, x, x)
        x = self.norm1(x + attn_out)
        x = self.adapter_after_attn(x)  # Adapter after attention
        
        # FFN + adapter
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        x = self.adapter_after_ffn(x)   # Adapter after FFN
        
        return x

# Demo Adapter
print("=== Adapter Layer Implementation ===\n")

# Single adapter layer
adapter = AdapterLayer(hidden_dim=256, bottleneck_dim=32)

adapter_params = sum(p.numel() for p in adapter.parameters())
print(f"Adapter architecture: {256} → {32} → {256}")
print(f"Adapter parameters: {adapter_params}")
print(f"  Down-proj: {256 * 32 + 32} = {256 * 32 + 32}")
print(f"  Up-proj: {32 * 256 + 256} = {32 * 256 + 256}")
print(f"  LayerNorm: {256 * 2} = {256 * 2}")
print()

# Test forward pass
x = torch.randn(2, 10, 256)
out = adapter(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {out.shape}")
print(f"Residual preserved: {torch.allclose(x, out, atol=0.1)}")
print()

# Transformer with adapters
print("=== Transformer Block with Adapters ===\n")

transformer_adapter = TransformerWithAdapters(hidden_dim=256, num_heads=4, bottleneck_dim=32)

# Count parameters
total_params = sum(p.numel() for p in transformer_adapter.parameters())
adapter_only = sum(p.numel() for p in transformer_adapter.adapter_after_attn.parameters()) + \
               sum(p.numel() for p in transformer_adapter.adapter_after_ffn.parameters())
frozen_params = total_params - adapter_only

print(f"Total parameters: {total_params:,}")
print(f"Frozen (original): {frozen_params:,}")
print(f"Trainable (adapters): {adapter_only:,}")
print(f"Adapter ratio: {adapter_only/total_params*100:.2f}%")

# Freeze original parameters, only train adapters
for name, param in transformer_adapter.named_parameters():
    if 'adapter' not in name:
        param.requires_grad = False

trainable = sum(p.numel() for p in transformer_adapter.parameters() if p.requires_grad)
print(f"\nAfter freezing:")
print(f"  Trainable parameters: {trainable:,}")
print(f"  Frozen parameters: {total_params - trainable:,}")

# Test forward pass
x = torch.randn(2, 10, 256)
out = transformer_adapter(x)
print(f"\nInput: {x.shape} → Output: {out.shape}")

print("\n观察：")
print("- Adapter只增加少量参数（~3%）")
print("- 残差连接确保初始时adapter不影响原始输出")
print("- 瓶颈结构有效压缩信息")
print("\n✓ Adapter层实现完成！")

## 5. Prefix-Tuning 和 Prompt-Tuning

### 5.1 Prefix-Tuning

**核心思想**：在每个Transformer层的Key和Value前面添加可训练的"前缀"向量。

**数学表示**：

原始注意力：
$$\text{Attention}(Q, K, V)$$

Prefix-Tuning后：
$$\text{Attention}(Q, [P_K; K], [P_V; V])$$

其中 $P_K, P_V \in \mathbb{R}^{l \times d}$ 是可训练的前缀参数，$l$ 是前缀长度。

**特点**：
- 每层都有独立的前缀参数
- 通过MLP重参数化提高训练稳定性
- 不修改原始模型参数

### 5.2 Prompt-Tuning

**核心思想**：在输入层添加可训练的"软提示"嵌入。

**数学表示**：

$$\text{input} = [P_1, P_2, ..., P_l, x_1, x_2, ..., x_n]$$

其中 $P_i \in \mathbb{R}^d$ 是可训练的软提示向量。

**与Prefix-Tuning的区别**：
- Prompt-Tuning只修改输入层
- Prefix-Tuning修改每一层
- Prompt-Tuning参数更少，但效果可能稍差
- 在大模型上（>10B参数），Prompt-Tuning效果接近全量微调

In [ ]:
# 🔬 Micro Practice: Implement Prefix-Tuning and Prompt-Tuning
# Goal: Build prefix and prompt tuning from scratch
# Expected outcome: Understand soft prompt methods

class PrefixTuning(nn.Module):
    """
    Prefix-Tuning: Prepend trainable prefix vectors to each layer's K and V
    """
    
    def __init__(self, num_layers, hidden_dim, prefix_length=10):
        """
        Args:
            num_layers: Number of transformer layers
            hidden_dim: Hidden dimension
            prefix_length: Number of prefix tokens
        """
        super(PrefixTuning, self).__init__()
        
        self.prefix_length = prefix_length
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        
        # Trainable prefix for each layer's key and value
        # Shape: (num_layers, 2, prefix_length, hidden_dim) - 2 for K and V
        self.prefix = nn.Parameter(
            torch.randn(num_layers, 2, prefix_length, hidden_dim) * 0.01
        )
        
        # Optional: reparameterization through MLP for stable training
        self.prefix_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.Tanh(),
            nn.Linear(hidden_dim * 2, hidden_dim)
        )
    
    def get_prefix(self, layer_idx, batch_size=1):
        """
        Get prefix K and V for a specific layer
        
        Args:
            layer_idx: Layer index
            batch_size: Batch size
        
        Returns:
            prefix_k: (batch_size, prefix_length, hidden_dim)
            prefix_v: (batch_size, prefix_length, hidden_dim)
        """
        prefix_k = self.prefix_mlp(self.prefix[layer_idx, 0])  # (prefix_length, hidden_dim)
        prefix_v = self.prefix_mlp(self.prefix[layer_idx, 1])
        
        # Expand for batch
        prefix_k = prefix_k.unsqueeze(0).expand(batch_size, -1, -1)
        prefix_v = prefix_v.unsqueeze(0).expand(batch_size, -1, -1)
        
        return prefix_k, prefix_v
    
    def forward(self, K, V, layer_idx, batch_size=1):
        """
        Prepend prefix to K and V
        
        Args:
            K: Original key (batch_size, seq_len, hidden_dim)
            V: Original value (batch_size, seq_len, hidden_dim)
            layer_idx: Current layer index
            batch_size: Batch size
        
        Returns:
            K_with_prefix: (batch_size, prefix_length + seq_len, hidden_dim)
            V_with_prefix: (batch_size, prefix_length + seq_len, hidden_dim)
        """
        prefix_k, prefix_v = self.get_prefix(layer_idx, batch_size)
        
        K_with_prefix = torch.cat([prefix_k, K], dim=1)
        V_with_prefix = torch.cat([prefix_v, V], dim=1)
        
        return K_with_prefix, V_with_prefix

class PromptTuning(nn.Module):
    """
    Prompt-Tuning: Prepend trainable soft prompt embeddings to input
    Simpler than prefix-tuning - only modifies input layer
    """
    
    def __init__(self, embedding_dim, prompt_length=20):
        """
        Args:
            embedding_dim: Dimension of embeddings
            prompt_length: Number of soft prompt tokens
        """
        super(PromptTuning, self).__init__()
        
        self.prompt_length = prompt_length
        
        # Trainable soft prompt embeddings
        self.soft_prompt = nn.Parameter(
            torch.randn(prompt_length, embedding_dim) * 0.01
        )
    
    def forward(self, input_embeddings):
        """
        Prepend soft prompt to input embeddings
        
        Args:
            input_embeddings: (batch_size, seq_len, embedding_dim)
        
        Returns:
            prompted_embeddings: (batch_size, prompt_length + seq_len, embedding_dim)
        """
        batch_size = input_embeddings.shape[0]
        
        # Expand prompt for batch
        prompt = self.soft_prompt.unsqueeze(0).expand(batch_size, -1, -1)
        
        # Concatenate prompt with input
        prompted = torch.cat([prompt, input_embeddings], dim=1)
        
        return prompted

# Demo Prefix-Tuning
print("=== Prefix-Tuning ===\n")

prefix_tuner = PrefixTuning(num_layers=4, hidden_dim=64, prefix_length=10)

# Simulate K, V from a transformer layer
batch_size = 2
seq_len = 20
K = torch.randn(batch_size, seq_len, 64)
V = torch.randn(batch_size, seq_len, 64)

K_prefixed, V_prefixed = prefix_tuner(K, V, layer_idx=0, batch_size=batch_size)

prefix_params = sum(p.numel() for p in prefix_tuner.parameters())
print(f"Prefix length: {prefix_tuner.prefix_length}")
print(f"Original K shape: {K.shape}")
print(f"Prefixed K shape: {K_prefixed.shape}")
print(f"Trainable parameters: {prefix_params}")
print()

# Demo Prompt-Tuning
print("=== Prompt-Tuning ===\n")

prompt_tuner = PromptTuning(embedding_dim=64, prompt_length=20)

# Simulate input embeddings
input_emb = torch.randn(batch_size, seq_len, 64)
prompted_emb = prompt_tuner(input_emb)

prompt_params = sum(p.numel() for p in prompt_tuner.parameters())
print(f"Prompt length: {prompt_tuner.prompt_length}")
print(f"Original input shape: {input_emb.shape}")
print(f"Prompted input shape: {prompted_emb.shape}")
print(f"Trainable parameters: {prompt_params}")
print()

# Compare
print("=== Comparison ===\n")
print(f"{'Method':<20} {'Parameters':<15} {'Modifies':<25}")
print("-" * 60)
print(f"{'Prefix-Tuning':<20} {prefix_params:<15} {'Every layer (K, V)':<25}")
print(f"{'Prompt-Tuning':<20} {prompt_params:<15} {'Input layer only':<25}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Prefix-Tuning diagram
axes[0].set_title('Prefix-Tuning', fontsize=14, weight='bold')
axes[0].axis('off')

for i in range(4):
    y = 0.8 - i * 0.2
    # Layer box
    axes[0].add_patch(plt.Rectangle((0.3, y-0.06), 0.4, 0.12, fill=True,
                     facecolor='lightblue', edgecolor='black', linewidth=1.5))
    axes[0].text(0.5, y, f'Layer {i+1}', ha='center', va='center', fontsize=10)
    
    # Prefix box
    axes[0].add_patch(plt.Rectangle((0.05, y-0.04), 0.2, 0.08, fill=True,
                     facecolor='lightgreen', edgecolor='black', linewidth=1))
    axes[0].text(0.15, y, f'Prefix', ha='center', va='center', fontsize=8, weight='bold')
    
    axes[0].annotate('', xy=(0.3, y), xytext=(0.25, y),
                    arrowprops=dict(arrowstyle='->', color='green', lw=1.5))

axes[0].text(0.5, 0.95, '每层都有可训练prefix', ha='center', fontsize=10, style='italic')

# Prompt-Tuning diagram
axes[1].set_title('Prompt-Tuning', fontsize=14, weight='bold')
axes[1].axis('off')

for i in range(4):
    y = 0.8 - i * 0.2
    axes[1].add_patch(plt.Rectangle((0.3, y-0.06), 0.4, 0.12, fill=True,
                     facecolor='lightblue', edgecolor='black', linewidth=1.5))
    axes[1].text(0.5, y, f'Layer {i+1}', ha='center', va='center', fontsize=10)

# Only input prompt
axes[1].add_patch(plt.Rectangle((0.05, 0.14), 0.2, 0.08, fill=True,
                 facecolor='lightyellow', edgecolor='black', linewidth=1))
axes[1].text(0.15, 0.18, 'Soft Prompt', ha='center', va='center', fontsize=8, weight='bold')
axes[1].annotate('', xy=(0.3, 0.18), xytext=(0.25, 0.18),
                arrowprops=dict(arrowstyle='->', color='orange', lw=1.5))

axes[1].text(0.5, 0.95, '只在输入层添加soft prompt', ha='center', fontsize=10, style='italic')

plt.tight_layout()
plt.show()

print("\n观察：")
print("- Prefix-Tuning：每层都有prefix，参数更多，效果更好")
print("- Prompt-Tuning：只修改输入，参数最少，实现最简单")
print("- 两者都不修改原始模型参数")
print("\n✓ Prefix和Prompt Tuning实现完成！")

## 6. 方法比较与分析

### 6.1 PEFT方法对比

| 方法 | 可训练参数 | 推理延迟 | 多任务 | 实现复杂度 |
|------|-----------|---------|--------|-----------|
| Full Fine-tune | 100% | 无额外 | 需要多个模型 | 简单 |
| LoRA | 0.1-1% | 无额外（可合并） | 切换adapter | 中等 |
| Adapter | 1-5% | 有额外 | 切换adapter | 中等 |
| Prefix-tuning | ~0.1% | 有额外 | 切换prefix | 较复杂 |
| Prompt-tuning | <0.1% | 有额外 | 切换prompt | 简单 |

### 6.2 选择指南

- **性能优先** → LoRA (r=8-16)
- **效率优先** → Prompt Tuning
- **多任务部署** → Adapter 或 LoRA
- **推理速度** → LoRA（权重可合并）
- **实现简单** → BitFit 或 Prompt Tuning

In [ ]:
# 🔬 Micro Practice: Comprehensive PEFT Comparison
# Goal: Benchmark all PEFT methods
# Expected outcome: Understand trade-offs between methods

class PEFTBenchmark:
    """
    Benchmark different PEFT methods on the same task
    """
    
    def __init__(self, input_dim=128, hidden_dim=256, output_dim=10):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        
        # Create base model (simulating pre-trained model)
        self.base_model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
        
        self.total_params = sum(p.numel() for p in self.base_model.parameters())
    
    def benchmark_full_finetune(self):
        """Full fine-tuning: all parameters trainable"""
        model = nn.Sequential(
            nn.Linear(self.input_dim, self.hidden_dim),
            nn.ReLU(),
            nn.Linear(self.hidden_dim, self.hidden_dim),
            nn.ReLU(),
            nn.Linear(self.hidden_dim, self.output_dim)
        )
        trainable = sum(p.numel() for p in model.parameters())
        return trainable, trainable
    
    def benchmark_lora(self, rank=4):
        """LoRA: low-rank adapters"""
        trainable = 0
        for name, param in self.base_model.named_parameters():
            if 'weight' in name and param.dim() == 2:
                d_out, d_in = param.shape
                trainable += rank * d_in + d_out * rank  # A + B matrices
        return trainable, self.total_params + trainable
    
    def benchmark_adapter(self, bottleneck=32):
        """Adapter: bottleneck layers"""
        trainable = 0
        # 2 adapter layers (after attention and after FFN in each transformer layer)
        # Each adapter: down_proj + up_proj + bias
        n_adapters = 2  # Simulating 2 adapter insertion points
        for _ in range(n_adapters):
            trainable += self.hidden_dim * bottleneck + bottleneck  # Down + bias
            trainable += bottleneck * self.hidden_dim + self.hidden_dim  # Up + bias
        return trainable, self.total_params + trainable
    
    def benchmark_prefix(self, prefix_length=10):
        """Prefix-tuning: trainable prefix vectors"""
        # Prefix for each layer's key and value
        n_layers = 2  # Simulating 2 layers
        trainable = prefix_length * self.hidden_dim * 2 * n_layers  # K and V prefixes
        return trainable, self.total_params + trainable
    
    def benchmark_prompt(self, prompt_length=20):
        """Prompt-tuning: trainable input embeddings"""
        trainable = prompt_length * self.input_dim
        return trainable, self.total_params + trainable
    
    def benchmark_bitfit(self):
        """BitFit: only bias parameters"""
        trainable = sum(p.numel() for name, p in self.base_model.named_parameters() if 'bias' in name)
        return trainable, self.total_params

# Run benchmark
bench = PEFTBenchmark(input_dim=128, hidden_dim=256, output_dim=10)

print("=== PEFT Methods Comprehensive Comparison ===\n")
print(f"Base model parameters: {bench.total_params:,}\n")

methods = {
    'Full Fine-tune': bench.benchmark_full_finetune(),
    'LoRA (r=4)': bench.benchmark_lora(rank=4),
    'LoRA (r=8)': bench.benchmark_lora(rank=8),
    'Adapter (b=32)': bench.benchmark_adapter(bottleneck=32),
    'Prefix (l=10)': bench.benchmark_prefix(prefix_length=10),
    'Prompt (l=20)': bench.benchmark_prompt(prompt_length=20),
    'BitFit': bench.benchmark_bitfit(),
}

print(f"{'Method':<20} {'Trainable':<15} {'Total':<15} {'Trainable %':<15}")
print("-" * 65)
for name, (trainable, total) in methods.items():
    pct = trainable / bench.total_params * 100
    print(f"{name:<20} {trainable:<15,} {total:<15,} {pct:<15.2f}")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

method_names = list(methods.keys())
trainable_params = [methods[m][0] for m in method_names]
trainable_pcts = [methods[m][0] / bench.total_params * 100 for m in method_names]
colors = ['red', 'blue', 'dodgerblue', 'green', 'orange', 'purple', 'gray']

# 1. Trainable parameters (absolute)
bars = axes[0].bar(range(len(method_names)), trainable_params, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_xticks(range(len(method_names)))
axes[0].set_xticklabels(method_names, rotation=45, ha='right', fontsize=8)
axes[0].set_title('Trainable Parameters', fontsize=12, weight='bold')
axes[0].set_ylabel('Count')
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3, axis='y')

# 2. Trainable percentage
bars = axes[1].bar(range(len(method_names)), trainable_pcts, color=colors, alpha=0.7, edgecolor='black')
axes[1].set_xticks(range(len(method_names)))
axes[1].set_xticklabels(method_names, rotation=45, ha='right', fontsize=8)
axes[1].set_title('Trainable Parameters (%)', fontsize=12, weight='bold')
axes[1].set_ylabel('Percentage')
axes[1].grid(True, alpha=0.3, axis='y')

for bar, pct in zip(bars, trainable_pcts):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'{pct:.1f}%', ha='center', va='bottom', fontsize=8, weight='bold')

# 3. Efficiency vs expected performance (conceptual)
# Simulated performance scores (based on literature)
perf_scores = [100, 97, 98, 95, 92, 88, 85]
efficiency = [1/p * 100 for p in trainable_pcts]

axes[2].scatter(trainable_pcts, perf_scores, c=colors, s=200, edgecolors='black', zorder=5)
for i, name in enumerate(method_names):
    short_name = name.split('(')[0].strip()
    axes[2].annotate(short_name, (trainable_pcts[i], perf_scores[i]),
                    textcoords="offset points", xytext=(5, 5), fontsize=8)

axes[2].set_xlabel('Trainable Parameters (%)', fontsize=10)
axes[2].set_ylabel('Expected Performance (%)', fontsize=10)
axes[2].set_title('Efficiency vs Performance', fontsize=12, weight='bold')
axes[2].set_xscale('log')
axes[2].grid(True, alpha=0.3)
axes[2].set_ylim(80, 105)

plt.tight_layout()
plt.show()

print("\n核心观察：")
print("- LoRA：最佳性能/效率平衡（~2%参数，~97%性能）")
print("- Adapter：参数稍多，但多任务灵活")
print("- Prefix/Prompt：参数最少，但性能有差距")
print("- BitFit：最简单，适合快速实验")
print("\n✓ PEFT方法综合比较完成！")

## 7. PEFT库使用与多任务部署

### 7.1 Hugging Face PEFT库

在实际项目中，推荐使用Hugging Face的PEFT库：

```python
# 安装: pip install peft transformers

from peft import LoraConfig, get_peft_model, TaskType

# 配置LoRA
config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,                        # LoRA rank
    lora_alpha=32,              # Scaling factor
    lora_dropout=0.1,           # Dropout
    target_modules=["query", "value"]  # 目标模块
)

# 应用到模型
model = get_peft_model(base_model, config)
model.print_trainable_parameters()

# 训练（与普通训练相同）
# optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
# ...

# 保存和加载
# model.save_pretrained("lora_checkpoint")
# model = PeftModel.from_pretrained(base_model, "lora_checkpoint")

# 合并权重（推理优化）
# merged_model = model.merge_and_unload()
```

### 7.2 多任务PEFT部署

**核心思想**：一个基础模型 + 多个轻量级adapter

**优势**：
- 存储高效：只需保存小型adapter
- 切换快速：加载不同adapter即可
- 独立训练：各任务adapter互不影响

# 🔬 Micro Practice: Using Hugging Face PEFT Library
# Goal: Learn to use PEFT in real projects
# Expected outcome: Master practical PEFT workflows

print("=== Hugging Face PEFT Library Usage ===\n")

print("PEFT库是Hugging Face提供的参数高效微调工具包")
print("支持LoRA, Prefix-Tuning, P-Tuning, Prompt-Tuning等多种方法\n")

print("安装命令:")
print("  pip install peft transformers\n")

print("="*70)
print("\n1. LoRA配置示例\n")
print("-"*70)

# 展示LoRA配置代码（注释形式，因为可能没有安装库）
lora_config_example = """
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForSequenceClassification

# 加载基础模型
base_model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# 配置LoRA
config = LoraConfig(
    task_type=TaskType.SEQ_CLS,  # 任务类型
    r=8,                          # LoRA rank
    lora_alpha=32,                # Scaling factor (通常设为rank的4倍)
    lora_dropout=0.1,             # Dropout概率
    target_modules=["query", "value"],  # 目标模块（Q和V投影）
    bias="none",                  # 是否训练bias
)

# 应用LoRA到模型
model = get_peft_model(base_model, config)

# 查看可训练参数
model.print_trainable_parameters()
# 输出示例: trainable params: 294,912 || all params: 109,483,778 || trainable%: 0.27%
"""

print(lora_config_example)

print("\n" + "="*70)
print("\n2. 训练流程\n")
print("-"*70)

training_example = """
import torch
from torch.utils.data import DataLoader
from transformers import AdamW, get_linear_schedule_with_warmup

# 准备数据加载器
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# 优化器（只优化可训练参数）
optimizer = AdamW(model.parameters(), lr=3e-4)

# 学习率调度器
num_training_steps = len(train_dataloader) * num_epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_training_steps // 10,
    num_training_steps=num_training_steps
)

# 训练循环（与普通训练相同）
model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        outputs = model(**batch)
        loss = outputs.loss
        
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
"""

print(training_example)

print("\n" + "="*70)
print("\n3. 保存和加载\n")
print("-"*70)

save_load_example = """
# 保存LoRA权重（只保存adapter，非常小！）
model.save_pretrained("./lora_checkpoint")
# 只会保存adapter_config.json和adapter_model.bin
# 文件大小通常只有几MB，而不是几GB

# 加载LoRA权重
from peft import PeftModel

base_model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased")
model = PeftModel.from_pretrained(base_model, "./lora_checkpoint")

# 推理模式
model.eval()
with torch.no_grad():
    outputs = model(**inputs)
"""

print(save_load_example)

print("\n" + "="*70)
print("\n4. 权重合并（推理优化）\n")
print("-"*70)

merge_example = """
# 合并LoRA权重到基础模型（消除推理延迟）
merged_model = model.merge_and_unload()

# 现在merged_model是一个标准的transformers模型
# 可以像普通模型一样使用，没有额外开销
merged_model.save_pretrained("./merged_model")

# 推理速度与原始模型相同
outputs = merged_model(**inputs)
"""

print(merge_example)

print("\n" + "="*70)
print("\n5. 多任务部署\n")
print("-"*70)

multitask_example = """
# 训练多个任务的adapter
# 任务1: 情感分析
model_sentiment = get_peft_model(base_model, lora_config)
# ... 训练 ...
model_sentiment.save_pretrained("./adapters/sentiment")

# 任务2: 命名实体识别
model_ner = get_peft_model(base_model, lora_config)
# ... 训练 ...
model_ner.save_pretrained("./adapters/ner")

# 任务3: 问答
model_qa = get_peft_model(base_model, lora_config)
# ... 训练 ...
model_qa.save_pretrained("./adapters/qa")

# 部署时：一个基础模型 + 多个小adapter
# 切换任务只需加载不同的adapter
base_model = AutoModel.from_pretrained("bert-base-uncased")

# 使用情感分析adapter
model = PeftModel.from_pretrained(base_model, "./adapters/sentiment")
# ... 推理 ...

# 切换到NER adapter
model = PeftModel.from_pretrained(base_model, "./adapters/ner")
# ... 推理 ...
"""

print(multitask_example)

print("\n" + "="*70)
print("\n6. 其他PEFT方法\n")
print("-"*70)

other_methods = """
# Prefix-Tuning
from peft import PrefixTuningConfig

prefix_config = PrefixTuningConfig(
    task_type=TaskType.SEQ_CLS,
    num_virtual_tokens=20,  # Prefix长度
    encoder_hidden_size=768
)

# Prompt-Tuning
from peft import PromptTuningConfig

prompt_config = PromptTuningConfig(
    task_type=TaskType.SEQ_CLS,
    num_virtual_tokens=20,  # Soft prompt长度
    prompt_tuning_init="TEXT",  # 初始化方式
)

# P-Tuning
from peft import PromptEncoderConfig

ptuning_config = PromptEncoderConfig(
    task_type=TaskType.SEQ_CLS,
    num_virtual_tokens=20,
    encoder_hidden_size=128
)
"""

print(other_methods)

print("\n" + "="*70)
print("\n实用建议：\n")
print("1. 首选LoRA：性能最好，使用最广泛")
print("2. rank选择：从8开始，简单任务用4，复杂任务用16-32")
print("3. lora_alpha：通常设为rank的2-4倍")
print("4. target_modules：BERT用['query', 'value']，GPT用['q_proj', 'v_proj']")
print("5. 学习率：PEFT通常需要更高的学习率（1e-4到3e-4）")
print("6. 多任务：共享基础模型，每个任务独立的adapter")
print("\n✓ PEFT库使用指南完成！")

In [ ]:
# 🔬 Micro Practice: Multi-task PEFT and Model Merging
# Goal: Demonstrate multi-task deployment with PEFT
# Expected outcome: Understand practical PEFT workflows

class MultiTaskPEFTModel:
    """
    Multi-task PEFT: One base model, multiple task-specific adapters
    """
    
    def __init__(self, base_model):
        self.base_model = base_model
        self.task_adapters = {}  # {task_name: adapter_weights}
        self.current_task = None
    
    def add_task(self, task_name, rank=4):
        """Add a new task-specific LoRA adapter"""
        adapters = {}
        for name, param in self.base_model.named_parameters():
            if 'weight' in name and param.dim() == 2:
                d_out, d_in = param.shape
                adapters[name] = {
                    'A': torch.randn(rank, d_in) * 0.01,
                    'B': torch.zeros(d_out, rank)
                }
        self.task_adapters[task_name] = adapters
        print(f"  Added adapter for task '{task_name}'")
    
    def switch_task(self, task_name):
        """Switch to a different task adapter"""
        if task_name in self.task_adapters:
            self.current_task = task_name
            print(f"  Switched to task '{task_name}'")
        else:
            print(f"  Task '{task_name}' not found!")
    
    def get_merged_weight(self, name):
        """Get weight with current task's LoRA applied"""
        base_weight = dict(self.base_model.named_parameters())[name]
        if self.current_task and name in self.task_adapters[self.current_task]:
            adapter = self.task_adapters[self.current_task][name]
            delta = adapter['B'] @ adapter['A']
            return base_weight.data + delta
        return base_weight.data
    
    def save_adapter(self, task_name, path):
        """Save only the adapter weights (very small!)"""
        adapter = self.task_adapters[task_name]
        total_params = sum(a['A'].numel() + a['B'].numel() for a in adapter.values())
        print(f"  Saving adapter '{task_name}': {total_params} parameters")
        # In practice: torch.save(adapter, path)
        return total_params

# Demo multi-task PEFT
print("=== Multi-task PEFT Deployment ===\n")

# Create base model (simulating a pre-trained model)
base = nn.Sequential(
    nn.Linear(256, 512),
    nn.ReLU(),
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Linear(256, 10)
)

multi_peft = MultiTaskPEFTModel(base)

# Add task-specific adapters
print("Adding task adapters:")
multi_peft.add_task('sentiment', rank=4)
multi_peft.add_task('ner', rank=8)
multi_peft.add_task('qa', rank=4)

print()

# Switch between tasks
print("Task switching:")
multi_peft.switch_task('sentiment')
multi_peft.switch_task('ner')
multi_peft.switch_task('qa')

print()

# Compare storage
base_params = sum(p.numel() for p in base.parameters())
print("Storage comparison:")
print(f"  Base model: {base_params:,} parameters")
for task in ['sentiment', 'ner', 'qa']:
    adapter_params = multi_peft.save_adapter(task, f'{task}_adapter.pt')
    print(f"    → {adapter_params/base_params*100:.1f}% of base model")

print()

# Visualize multi-task architecture
fig, ax = plt.subplots(figsize=(12, 6))
ax.axis('off')

# Draw base model
ax.add_patch(plt.Rectangle((0.3, 0.3), 0.4, 0.4, fill=True, 
             facecolor='lightblue', edgecolor='black', linewidth=2))
ax.text(0.5, 0.5, 'Base Model\n(Frozen)\n' + f'{base_params:,} params', 
        ha='center', va='center', fontsize=12, weight='bold')

# Draw adapters
tasks = ['Sentiment\n(rank=4)', 'NER\n(rank=8)', 'QA\n(rank=4)']
colors = ['lightgreen', 'lightyellow', 'lightsalmon']
positions = [0.1, 0.4, 0.7]

for i, (task, color, pos) in enumerate(zip(tasks, colors, positions)):
    ax.add_patch(plt.Rectangle((pos, 0.78), 0.2, 0.15, fill=True,
                 facecolor=color, edgecolor='black', linewidth=1.5))
    ax.text(pos + 0.1, 0.855, task, ha='center', va='center', fontsize=9, weight='bold')
    ax.annotate('', xy=(0.5, 0.7), xytext=(pos + 0.1, 0.78),
               arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

ax.text(0.5, 0.98, 'Multi-task PEFT: One Base Model + Multiple Adapters',
        ha='center', va='top', fontsize=14, weight='bold')
ax.text(0.5, 0.2, '优势：共享基础模型，每个任务只需存储小型adapter',
        ha='center', va='center', fontsize=11, style='italic')

ax.set_xlim(0, 1)
ax.set_ylim(0.1, 1.0)
plt.tight_layout()
plt.show()

print("\n✓ 多任务PEFT部署演示完成！")

## 9. 高级主题与总结

### 9.1 高级PEFT方法

**QLoRA (Quantized LoRA)**：
- 将基础模型量化到4-bit
- 在量化模型上应用LoRA
- 极大减少内存需求（可在单GPU上微调65B模型）

**(IA)³ (Infused Adapter by Inhibiting and Amplifying Inner Activations)**：
- 学习缩放向量而非矩阵
- 分别缩放K、V和FFN的激活
- 参数量极少（每层只需几百个参数）

**BitFit (Bias-only Fine-tuning)**：
- 只训练模型中的偏置参数
- 最简单的PEFT方法
- 适合资源极度受限的场景

## 8. 实际应用案例

### 8.1 案例概览

PEFT在实际项目中的典型应用场景：

1. **情感分析**：电商评论、社交媒体情感分类
2. **命名实体识别（NER）**：医疗、法律、金融领域的实体提取
3. **问答系统（QA）**：客服机器人、知识问答
4. **文本摘要**：新闻摘要、文档总结
5. **机器翻译**：特定领域的翻译任务

### 8.2 应用案例对比

| 任务 | 推荐方法 | Rank | 训练数据 | 预期性能 |
|------|---------|------|---------|---------|
| 情感分析 | LoRA | 4-8 | 5K-50K | 95%+ |
| NER | LoRA | 8-16 | 10K-100K | 90%+ |
| QA | LoRA + Prefix | 16-32 | 50K-500K | 85%+ |
| 摘要 | LoRA | 16-32 | 100K+ | 80%+ |
| 翻译 | LoRA | 32-64 | 500K+ | 85%+ |

### 8.3 实施步骤

```
1. 选择预训练模型（BERT, RoBERTa, GPT等）
   ↓
2. 配置PEFT方法（LoRA推荐）
   ↓
3. 准备任务数据集
   ↓
4. 微调训练（2-10 epochs）
   ↓
5. 评估和优化
   ↓
6. 部署（合并权重或保留adapter）
```

# 🔬 Micro Practice: Real-world Application Examples
# Goal: Understand PEFT in practical scenarios
# Expected outcome: Know how to apply PEFT to real tasks

print("=== PEFT实际应用案例 ===\n")

# 案例1: 情感分析
print("="*70)
print("\n案例1: 电商评论情感分析\n")
print("-"*70)

sentiment_example = """
任务描述：
- 对电商产品评论进行情感分类（正面/负面/中性）
- 数据集：10,000条标注评论
- 基础模型：BERT-base-chinese

PEFT配置：
```python
from peft import LoraConfig, get_peft_model, TaskType
from transformers import BertForSequenceClassification

# 加载模型
model = BertForSequenceClassification.from_pretrained(
    "bert-base-chinese",
    num_labels=3  # 正面、负面、中性
)

# LoRA配置
config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=4,                    # 简单任务，rank=4足够
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"]
)

model = get_peft_model(model, config)
# trainable params: 147,456 || all params: 102,267,648 || trainable%: 0.14%
```

训练配置：
- Batch size: 32
- Learning rate: 3e-4
- Epochs: 5
- 训练时间: ~30分钟（单GPU）

结果：
- 准确率: 94.2%
- F1-score: 93.8%
- 与全量微调相比: -0.5%性能，但训练时间减少60%
"""

print(sentiment_example)

# 案例2: 命名实体识别
print("\n" + "="*70)
print("\n案例2: 医疗文本命名实体识别\n")
print("-"*70)

ner_example = """
任务描述：
- 从医疗文本中识别疾病、症状、药物等实体
- 数据集：50,000条标注句子
- 基础模型：BioBERT

PEFT配置：
```python
from transformers import BertForTokenClassification

# 加载医疗领域预训练模型
model = BertForTokenClassification.from_pretrained(
    "dmis-lab/biobert-v1.1",
    num_labels=9  # B-DIS, I-DIS, B-SYM, I-SYM, B-MED, I-MED, O等
)

# LoRA配置（NER任务需要更高的rank）
config = LoraConfig(
    task_type=TaskType.TOKEN_CLS,
    r=8,                    # NER任务用rank=8
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "value", "key"]  # NER建议包含key
)

model = get_peft_model(model, config)
# trainable params: 442,368 || all params: 108,895,233 || trainable%: 0.41%
```

训练配置：
- Batch size: 16
- Learning rate: 2e-4
- Epochs: 8
- 训练时间: ~2小时（单GPU）

结果：
- F1-score: 91.5%
- 实体级准确率: 89.3%
- 与全量微调相比: -1.2%性能，内存使用减少75%
"""

print(ner_example)

# 案例3: 问答系统
print("\n" + "="*70)
print("\n案例3: 客服问答系统\n")
print("-"*70)

qa_example = """
任务描述：
- 基于上下文回答客户问题
- 数据集：100,000个问答对
- 基础模型：RoBERTa-large

PEFT配置：
```python
from transformers import RobertaForQuestionAnswering

# 加载模型
model = RobertaForQuestionAnswering.from_pretrained("roberta-large")

# LoRA配置（QA任务需要较高rank）
config = LoraConfig(
    task_type=TaskType.QUESTION_ANS,
    r=16,                   # 复杂任务用rank=16
    lora_alpha=64,
    lora_dropout=0.1,
    target_modules=["query", "value"]
)

model = get_peft_model(model, config)
# trainable params: 1,179,648 || all params: 355,359,744 || trainable%: 0.33%
```

训练配置：
- Batch size: 8
- Learning rate: 1e-4
- Epochs: 3
- 训练时间: ~6小时（单GPU）

结果：
- EM (Exact Match): 78.5%
- F1-score: 86.2%
- 与全量微调相比: -2.1%性能，但可在单GPU上训练（全量需要多GPU）
"""

print(qa_example)

# 案例对比
print("\n" + "="*70)
print("\n案例对比总结\n")
print("-"*70)

comparison_data = {
    '情感分析': {
        'task_complexity': '简单',
        'rank': 4,
        'trainable_params': '147K',
        'training_time': '30分钟',
        'performance_gap': '-0.5%',
        'memory_saving': '80%'
    },
    'NER': {
        'task_complexity': '中等',
        'rank': 8,
        'trainable_params': '442K',
        'training_time': '2小时',
        'performance_gap': '-1.2%',
        'memory_saving': '75%'
    },
    'QA': {
        'task_complexity': '复杂',
        'rank': 16,
        'trainable_params': '1.18M',
        'training_time': '6小时',
        'performance_gap': '-2.1%',
        'memory_saving': '70%'
    }
}

print(f"{'任务':<12} {'复杂度':<10} {'Rank':<8} {'可训练参数':<12} {'训练时间':<12} {'性能差距':<12} {'内存节省':<10}")
print("-"*80)
for task, data in comparison_data.items():
    print(f"{task:<12} {data['task_complexity']:<10} {data['rank']:<8} {data['trainable_params']:<12} "
          f"{data['training_time']:<12} {data['performance_gap']:<12} {data['memory_saving']:<10}")

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

tasks = list(comparison_data.keys())
ranks = [comparison_data[t]['rank'] for t in tasks]
params = [float(comparison_data[t]['trainable_params'].replace('K', '').replace('M', '000')) for t in tasks]
gaps = [abs(float(comparison_data[t]['performance_gap'].replace('%', ''))) for t in tasks]
colors = ['lightgreen', 'lightyellow', 'lightcoral']

# Rank vs Task Complexity
axes[0].bar(tasks, ranks, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_ylabel('LoRA Rank', fontsize=11)
axes[0].set_title('Recommended Rank by Task', fontsize=12, weight='bold')
axes[0].grid(True, alpha=0.3, axis='y')
for i, (task, rank) in enumerate(zip(tasks, ranks)):
    axes[0].text(i, rank, f'r={rank}', ha='center', va='bottom', fontsize=10, weight='bold')

# Trainable Parameters
axes[1].bar(tasks, params, color=colors, alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Trainable Parameters (K)', fontsize=11)
axes[1].set_title('Parameter Efficiency', fontsize=12, weight='bold')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3, axis='y')

# Performance Gap
axes[2].bar(tasks, gaps, color=colors, alpha=0.7, edgecolor='black')
axes[2].set_ylabel('Performance Gap (%)', fontsize=11)
axes[2].set_title('Performance vs Full Fine-tuning', fontsize=12, weight='bold')
axes[2].grid(True, alpha=0.3, axis='y')
axes[2].set_ylim(0, 3)
for i, (task, gap) in enumerate(zip(tasks, gaps)):
    axes[2].text(i, gap, f'-{gap}%', ha='center', va='bottom', fontsize=10, weight='bold')

plt.tight_layout()
plt.show()

print("\n核心观察：")
print("1. 任务越复杂，需要的rank越高")
print("2. 即使是复杂任务，PEFT也只需要<1%的参数")
print("3. 性能损失通常在1-2%以内，可接受")
print("4. 内存节省显著，使得大模型微调成为可能")
print("5. 训练时间大幅减少，加快迭代速度")

print("\n实施建议：")
print("- 从小rank开始（4或8），逐步增加直到性能满意")
print("- 简单任务（分类）用rank=4-8")
print("- 中等任务（NER、槽填充）用rank=8-16")
print("- 复杂任务（QA、生成）用rank=16-32")
print("- 监控验证集性能，避免过拟合")

print("\n✓ 实际应用案例完成！")

In [ ]:
# 🔬 Micro Practice: Advanced PEFT Methods
# Goal: Understand QLoRA, (IA)³, and BitFit
# Expected outcome: Know the full PEFT landscape

class BitFitModel(nn.Module):
    """
    BitFit: Only train bias parameters
    Simplest PEFT method - just unfreeze biases
    """
    
    def __init__(self, base_model):
        super(BitFitModel, self).__init__()
        self.model = base_model
        
        # Freeze all parameters
        for param in self.model.parameters():
            param.requires_grad = False
        
        # Unfreeze only bias parameters
        for name, param in self.model.named_parameters():
            if 'bias' in name:
                param.requires_grad = True
    
    def forward(self, x):
        return self.model(x)

class IA3Layer(nn.Module):
    """
    (IA)³: Infused Adapter by Inhibiting and Amplifying Inner Activations
    Learns scaling vectors for keys, values, and FFN
    """
    
    def __init__(self, in_features):
        super(IA3Layer, self).__init__()
        # Learnable scaling vector (initialized to ones)
        self.scaling = nn.Parameter(torch.ones(in_features))
    
    def forward(self, x):
        return x * self.scaling

class SimulatedQLoRA(nn.Module):
    """
    Simplified QLoRA simulation
    In practice: 4-bit quantized base model + LoRA adapters in fp16
    Here we simulate the concept with reduced precision
    """
    
    def __init__(self, in_features, out_features, rank=4):
        super(SimulatedQLoRA, self).__init__()
        
        # Simulate quantized base weight (reduced to int8 for demo)
        base_weight = torch.randn(out_features, in_features)
        # Quantize to int8 range and store
        self.scale = base_weight.abs().max() / 127
        self.quantized_weight = nn.Parameter(
            torch.round(base_weight / self.scale).clamp(-128, 127),
            requires_grad=False
        )
        
        # LoRA adapters in full precision
        self.lora_A = nn.Parameter(torch.randn(rank, in_features) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))
    
    def forward(self, x):
        # Dequantize base weight
        base_out = x @ (self.quantized_weight.float() * self.scale).T
        # Add LoRA
        lora_out = x @ self.lora_A.T @ self.lora_B.T
        return base_out + lora_out

print("=== Advanced PEFT Methods ===\n")

# 1. BitFit demo
print("1. BitFit (Bias-only Fine-tuning)")
print("-" * 40)

# Create a simple model
simple_model = nn.Sequential(
    nn.Linear(64, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10)
)

bitfit = BitFitModel(simple_model)

total_params = sum(p.numel() for p in bitfit.model.parameters())
trainable_params = sum(p.numel() for p in bitfit.model.parameters() if p.requires_grad)
print(f"  Total parameters: {total_params}")
print(f"  Trainable (bias only): {trainable_params}")
print(f"  Ratio: {trainable_params/total_params*100:.2f}%\n")

# 2. (IA)³ demo
print("2. (IA)³ (Learned Activation Scaling)")
print("-" * 40)

ia3_k = IA3Layer(64)
ia3_v = IA3Layer(64)
ia3_ff = IA3Layer(128)

ia3_params = sum(p.numel() for p in [ia3_k.scaling, ia3_v.scaling, ia3_ff.scaling])
print(f"  Scaling vectors: K({64}), V({64}), FFN({128})")
print(f"  Total trainable: {ia3_params}")
print(f"  Extremely parameter-efficient!\n")

# 3. QLoRA demo
print("3. QLoRA (Quantized LoRA)")
print("-" * 40)

qlora = SimulatedQLoRA(64, 128, rank=4)
qlora_trainable = sum(p.numel() for p in qlora.parameters() if p.requires_grad)
qlora_total = sum(p.numel() for p in qlora.parameters())

print(f"  Base weight: quantized (int8 simulation)")
print(f"  LoRA rank: 4")
print(f"  Trainable parameters: {qlora_trainable}")
print(f"  Memory savings: ~4x from quantization + LoRA efficiency")

# Test forward pass
x = torch.randn(2, 64)
out = qlora(x)
print(f"  Input: {x.shape} → Output: {out.shape}")

print("\n" + "="*50)
print("\n方法对比总结：")
print(f"{'Method':<15} {'Trainable %':<15} {'Memory':<15} {'Performance':<15}")
print("-" * 60)
print(f"{'Full FT':<15} {'100%':<15} {'Very High':<15} {'Best':<15}")
print(f"{'LoRA':<15} {'0.1-1%':<15} {'Low':<15} {'Near Best':<15}")
print(f"{'Adapter':<15} {'1-5%':<15} {'Medium':<15} {'Good':<15}")
print(f"{'Prefix':<15} {'0.1%':<15} {'Low':<15} {'Good':<15}")
print(f"{'Prompt':<15} {'<0.1%':<15} {'Very Low':<15} {'Moderate':<15}")
print(f"{'BitFit':<15} {'<0.1%':<15} {'Very Low':<15} {'Moderate':<15}")
print(f"{'(IA)³':<15} {'<0.01%':<15} {'Minimal':<15} {'Good':<15}")
print(f"{'QLoRA':<15} {'0.1-1%':<15} {'Minimal':<15} {'Near Best':<15}")

print("\n✓ 高级PEFT方法介绍完成！")

### 8.2 常见问题与调试

#### Q1: LoRA和Adapter哪个更好？

**A:** 取决于场景：
- **LoRA**：推理无额外延迟（权重可合并），参数更少
- **Adapter**：多任务切换更灵活，不同任务用不同Adapter
- 一般推荐先试LoRA

#### Q2: 如何选择LoRA的rank r？

**A:**
- r=4：最小配置，适合简单任务
- r=8：常用默认值，大多数任务够用
- r=16-64：复杂任务或大模型
- 经验法则：从r=8开始，根据性能调整

#### Q3: PEFT能达到全量微调的效果吗？

**A:**
- 在大多数任务上，PEFT能达到全量微调95%+的性能
- 某些复杂任务可能有差距
- 数据量越大，差距越小
- LoRA在很多benchmark上已经匹配全量微调

#### Q4: 如何调试PEFT训练？

**A:**
- 检查可训练参数是否正确（只有PEFT参数）
- 学习率通常比全量微调高（1e-4 到 3e-4）
- 监控训练损失是否下降
- 如果效果差，增加rank或训练更多epoch

#### Q5: 多个PEFT模块能组合使用吗？

**A:**
- 可以！例如LoRA + Prompt Tuning
- 但需要注意参数量和训练稳定性
- 实践中单一方法通常足够

### 9.3 本章总结

#### 核心要点

1. **PEFT动机**
   - 全量微调代价高昂（内存、计算、存储）
   - PEFT只训练少量参数（0.1%-10%）
   - 性能接近甚至匹配全量微调

2. **LoRA**
   - 低秩分解：$W' = W + BA$
   - 应用于注意力层的Q、V投影
   - 最流行的PEFT方法，推理无额外开销

3. **Adapter**
   - 瓶颈结构插入Transformer层间
   - 残差连接保持原始能力
   - 适合多任务场景

4. **Prefix/Prompt Tuning**
   - 在输入或每层添加可训练向量
   - 不修改模型结构
   - 参数最少的方法

5. **高级方法**
   - QLoRA：量化 + LoRA，极致效率
   - (IA)³：学习激活缩放因子
   - BitFit：只训练偏置项

6. **实际应用**
   - 情感分析：rank=4-8，性能损失<1%
   - NER：rank=8-16，内存节省75%
   - QA：rank=16-32，单GPU可训练大模型

#### 方法选择指南

```
需要最佳性能？ → LoRA (r=8-64)
需要多任务部署？ → Adapter
内存极度受限？ → QLoRA
最简单的方法？ → Prompt Tuning
推理速度优先？ → LoRA（可合并权重）
```

#### 实践建议

1. **首选LoRA**：最佳性能/效率平衡
2. **从小rank开始**：r=4或8，逐步增加
3. **关注目标模块**：Q和V投影通常最重要
4. **学习率调整**：PEFT通常需要更高的学习率（1e-4到3e-4）
5. **评估充分**：在多个任务上验证
6. **使用PEFT库**：Hugging Face PEFT简化实现
7. **监控性能**：确保PEFT效果满足需求

#### 性能对比总结

| 维度 | 全量微调 | LoRA | Adapter | Prompt |
|------|---------|------|---------|--------|
| 参数量 | 100% | 0.1-1% | 1-5% | <0.1% |
| 训练速度 | 基准 | 1.2-1.5x | 1.2-1.5x | 1.5-2x |
| 推理速度 | 基准 | 相同（合并后） | 稍慢 | 稍慢 |
| 内存使用 | 高 | 低 | 低 | 极低 |
| 性能 | 最佳 | 接近最佳 | 良好 | 中等 |
| 多任务 | 需多模型 | 易切换 | 易切换 | 易切换 |

### 9.4 下一章预告

**Module 5.3: 领域适应与持续学习**
- 领域自适应预训练（DAPT/TAPT）
- 灾难性遗忘问题
- 持续学习策略（EWC、Experience Replay）
- 多领域学习架构

### 💡 思考题

1. 为什么LoRA选择低秩分解而不是稀疏矩阵？
   > 提示：考虑训练稳定性和硬件效率

2. 如果rank r等于原始维度d，LoRA等价于什么？
   > 提示：此时低秩约束消失

3. Adapter和LoRA能否同时使用？效果如何？
   > 提示：可以组合，但需权衡复杂度

4. 为什么Prompt Tuning在小模型上效果不好？
   > 提示：与模型容量和表达能力有关

5. QLoRA如何在4-bit量化下保持训练精度？
   > 提示：关键在于梯度计算的精度

6. 在实际项目中，如何选择合适的LoRA rank？
   > 提示：从小开始，根据验证集性能调整

7. 多任务PEFT部署时，如何处理任务间的干扰？
   > 提示：独立adapter vs 共享部分参数

### 📚 扩展阅读

**论文**：
- LoRA: [Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685)
- QLoRA: [Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314)
- Adapter: [Parameter-Efficient Transfer Learning for NLP](https://arxiv.org/abs/1902.00751)
- Prefix-Tuning: [Optimizing Continuous Prompts for Generation](https://arxiv.org/abs/2101.00190)

**代码库**：
- [Hugging Face PEFT](https://github.com/huggingface/peft)
- [Adapter-Transformers](https://github.com/adapter-hub/adapter-transformers)

**博客**：
- [Hugging Face PEFT文档](https://huggingface.co/docs/peft)
- [LoRA详解](https://huggingface.co/blog/lora)

## 思考题参考答案

### 问题 1：为什么不能总是使用全量微调？全量微调的局限性体现在哪些方面？

**全量微调（Full Fine-tuning）的核心局限性：**

**1. 内存开销巨大**

训练时GPU需要存储：模型参数 + 梯度 + 优化器状态（Adam需要一阶+二阶矩）

$$\text{内存} \approx \underbrace{\Phi}_{\text{参数}} + \underbrace{\Phi}_{\text{梯度}} + \underbrace{2\Phi}_{\text{Adam状态}} = 4\Phi$$

| 模型 | 参数量 | FP16 参数 | 全量微调内存 |
|------|--------|-----------|------------|
| BERT-base | 110M | 0.2GB | ~1GB |
| LLaMA-7B | 7B | 14GB | ~56GB |
| LLaMA-65B | 65B | 130GB | >520GB |

LLaMA-65B 的全量微调需要 8 块 A100 (80GB) 以上，普通研究者完全无法承担。

**2. 存储成本高**

每个任务需要保存完整模型副本：
```
10个任务 × LLaMA-7B (14GB) = 140GB 存储
使用PEFT：10个任务 × LoRA增量 (14MB) = 140MB 存储  节省 1000倍！
```

**3. 灾难性遗忘风险**

全量微调会修改所有参数，在小数据集上容易过拟合，并破坏预训练时学到的通用知识。

**4. 训练时间长**

更新 70 亿个参数比更新 700 万个参数慢得多，当资源有限时这是不可接受的。

**5. 多任务部署困难**

全量微调无法实现参数共享，每个任务都需要独立部署，系统复杂度随任务数线性增加。

---

### 问题 2：LoRA 为什么有效？低秩假设的依据是什么？

**LoRA 的核心假设：预训练模型在适配新任务时，权重的变化量 $\Delta W$ 是低秩的。**

**数学表示：**

$$W' = W_0 + \Delta W = W_0 + BA$$

其中：
- $W_0 \in \mathbb{R}^{d \times k}$：预训练权重（冻结）
- $B \in \mathbb{R}^{d \times r}$，$A \in \mathbb{R}^{r \times k}$：低秩矩阵（$r \ll \min(d, k)$）

**参数节省量：**
$$\text{原参数量}: d \times k \qquad \text{LoRA参数量}: r \times (d + k)$$

以 GPT-3 的注意力层（$d = 12288$）为例，$r=4$ 时参数节省 **1536倍**。

**低秩假设的理论依据：**

1. **过参数化理论（Overparameterization）**：预训练时模型远比任务所需复杂，大量参数是"冗余"的。Aghajanyan et al. (2021) 发现预训练模型在微调时的本征秩（intrinsic rank）非常低——BERT 仅需 **200 个维度**就能表达大部分微调变化。

2. **矩阵奇异值分析**：对多个任务上的 $\Delta W$ 进行 SVD 分解，发现绝大多数奇异值接近于零，只有少数主导奇异值较大，这直接验证了低秩假设。

3. **神经网络的信息瓶颈**：有效的信息传递路径远少于全连接的潜在路径数量，任务适配只需修改少数关键方向。

**推理时零额外开销：**

$$W' = W_0 + BA$$

推理前将 $BA$ 合并到 $W_0$ 中，推理速度与原始模型完全相同，这是 LoRA 相比 Adapter 的重要优势。

---

### 问题 3：不同 PEFT 方法的参数效率和性能如何权衡？

**各方法对比矩阵：**

| 方法 | 参数量 | 性能 | 推理开销 | 训练速度 | 适用场景 |
|------|--------|------|---------|---------|---------|
| 全量微调 | 100% | ★★★★★ | 无 | 最慢 | 资源充足 |
| LoRA | 0.1-1% | ★★★★☆ | 无（合并后）| 较快 | 最常用 |
| Adapter | 1-5% | ★★★★☆ | 有（+延迟）| 较快 | 多任务共存 |
| Prefix-Tuning | 0.1-1% | ★★★☆☆ | 有（序列变长）| 快 | 生成任务 |
| Prompt-Tuning | <0.1% | ★★★☆☆ | 有（序列变长）| 最快 | 超大模型 |
| BitFit | <0.1% | ★★★☆☆ | 无 | 最快 | 快速原型 |

**选择指南（以性能优先级排序）：**

**场景 A：追求最高性能，资源允许 → LoRA (rank=16-64)**
- GLUE 基准上 LoRA-BERT 接近全量微调的 99% 性能
- 参数量仅为全量微调的 0.5%

**场景 B：多任务同时服务 → Adapter**
- 不同任务的 Adapter 模块独立，基础模型共享
- 运行时可动态切换任务

**场景 C：模型极大（>100B参数）→ Prompt-Tuning**
- Lester et al. (2021) 证明在 T5-11B 上 Prompt-Tuning 接近全量微调
- 只需训练 1000-10000 个 soft token

**场景 D：快速实验 → BitFit**
- 只训练偏置参数（<0.1% 参数量）
- 适合快速验证任务可行性

**性能 vs 参数效率权衡曲线：**
```
性能
  |          全量微调
  |        /
  |      LoRA / Adapter
  |    /
  |  Prefix-Tuning
  | Prompt-Tuning / BitFit
  |_________________________ 参数量（%）
  0     0.1    1    10   100
```

---

### 问题 4：实际项目中如何选择 PEFT 方法？给出完整决策流程。

**实际项目的 PEFT 方法选择决策树：**

```
Step 1: 确认模型规模
├── 模型 < 7B 参数 → 考虑全量微调或 LoRA
└── 模型 ≥ 7B 参数 → 必须使用 PEFT

Step 2: 确认部署需求
├── 需要同时支持多任务 → Adapter（模块可热插拔）
├── 推理延迟严格要求 → LoRA（合并后无开销）
└── 单任务部署 → LoRA 或 Prefix-Tuning

Step 3: 确认数据量
├── > 10000 样本 → LoRA（rank=8-16）
├── 1000-10000 样本 → LoRA（rank=4-8）
└── < 1000 样本 → Prompt-Tuning 或 LoRA（rank=4）

Step 4: 确认硬件资源
├── 单 GPU（≤24GB）→ LoRA + 4bit 量化（QLoRA）
├── 多 GPU → LoRA 或 全量微调
└── 极度受限 → BitFit + Gradient Checkpointing
```

**实际工程最佳实践：**

**首选 LoRA 的 7 个理由：**
1. 推理时可合并权重，零额外延迟
2. Hugging Face PEFT 库原生支持，接入成本低
3. 与量化（QLoRA）兼容，可在 24GB GPU 上运行 65B 模型
4. rank 超参数提供灵活的性能-效率权衡
5. 支持 RLHF（InstructGPT、Alpaca 均使用）
6. 社区生态最成熟，调参经验丰富
7. 任务切换只需替换小的增量权重（几十MB vs 数十GB）

**LoRA 关键超参数调优建议：**

| 超参数 | 推荐范围 | 影响 |
|--------|---------|------|
| rank (r) | 4-64 | 越大性能越高，参数越多 |
| alpha (α) | 通常 = 2r 或 r | 缩放因子，影响学习率 |
| target modules | q, v（注意力层） | 也可包含 k, o, FFN |
| dropout | 0.05-0.1 | 防过拟合 |

**快速开始代码（Hugging Face PEFT）：**
```python
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=16,                        # rank
    lora_alpha=32,               # alpha = 2r
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, config)
model.print_trainable_parameters()
# trainable params: 4,194,304 || all params: 6,742,609,920 || trainable%: 0.0622
```